In [ ]:
!git clone -b kaggle-implementation --single-branch https://github.com/nhanbayern/1003_EPA-Project_UIT.git
import sys
import os
sys.path.append('/kaggle/working/1003_EPA-Project_UIT/transformer based')
os.makedirs('/kaggle/working/results/all_predictions', exist_ok=True)
os.makedirs('/kaggle/working/results/metrics', exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from dataset import VolatilityDataset
from models import VanillaTransformer, Autoformer, Informer, Reformer
from utils import calculate_fixed_nu, StudentTNLLLoss, calc_mse, calc_mae, calc_qlike
import glob


In [ ]:
DATA_DIR = '/kaggle/input/datasets/trnhngv/historical-price'
HORIZONS = [1, 3, 5, 10, 21]
EVAL_INDICES = [0, 2, 4, 9, 20] # Indices corresponding to the horizons in the 21-length output
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
def evaluate_model(model, dataloader):
    model.eval()
    preds, targets_vol = [], []
    with torch.no_grad():
        for x, y_vol, _ in dataloader:
            x = x.to(DEVICE)
            pred_vol = model(x)
            preds.append(pred_vol.cpu().numpy())
            targets_vol.append(y_vol.numpy())
    return np.concatenate(preds, axis=0), np.concatenate(targets_vol, axis=0)


In [ ]:
def train_model(model, train_loader, val_loader, nu):
    criterion = StudentTNLLLoss(nu=nu)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    best_val_loss = float('inf')
    patience = 5
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for x, _, y_ret in train_loader:
            x, y_ret = x.to(DEVICE), y_ret.to(DEVICE)
            optimizer.zero_grad()
            pred_vol = model(x)
            loss = criterion(pred_vol, y_ret)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        # Simple validation step based on NLL
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, _, y_ret in val_loader:
                x, y_ret = x.to(DEVICE), y_ret.to(DEVICE)
                pred_vol = model(x)
                val_loss += criterion(pred_vol, y_ret).item()
                
        val_loss /= len(val_loader)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # Save best weights optionally
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch}')
                break


In [ ]:
# MAIN EXPERIMENT LOOP MOCKUP
print("Ready to implement the full training loop over all 9 indices and 4 models.")
